# 7.9 · 概率校准 / Probability Calibration

> **课程定位 / Where this fits**
> 第 9 课，**Part 7 · 模型评估与优化**。
> Lesson 9, **Part 7 · Model Evaluation & Tuning**.
>
> 模型说"违约概率 0.9"，这 0.9 **可信吗**？很多模型（SVM、随机森林、boosting、朴素贝叶斯）排序能力强，但输出的"概率"**数值不准**——0.9 可能实际只有 70%。在**风控、定价、医疗**等要用概率做**期望损失决策**(2.10)的场景，校准是硬需求。这一课把 4.16/5.15 的校准内容**系统化**：可靠性曲线、Brier 分数、Platt vs 等张校准。
> A model says "default probability 0.9" — **is that 0.9 trustworthy**? Many models (SVM, random forests, boosting, naive Bayes) rank well but their "probabilities" are **numerically off** — 0.9 may really be 70%. In **risk, pricing, medicine** where probabilities drive **expected-loss decisions** (2.10), calibration is a hard requirement. This lesson **systematizes** the calibration from 4.16/5.15: reliability curves, Brier score, Platt vs isotonic.
>
> 💼 **实战/面试视角**："什么是概率校准 / 哪些模型概率不准 / Platt vs 等张" 在风控/概率预测岗常考。
> 💼 **Practical/interview angle:** "what is calibration / which models are miscalibrated / Platt vs isotonic" — risk/probabilistic-forecasting roles.

> 💡 **面试相关 / Interview-relevant**
> - "什么是概率校准 / 可靠性曲线怎么读"（出镜率 ★★★★★）
> - "为什么 SVM/随机森林概率不准"（★★★★）
> - "Platt scaling vs 等张校准 区别"（★★★★）
> - "Brier 分数衡量什么"（★★★）
> - "校准为什么要独立校准集（防泄漏）"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解什么是"校准良好"+ 用可靠性曲线诊断。
   Understand "well-calibrated" and diagnose with a reliability curve.
2. 看清不同模型的校准差异（LR 好、RF/boosting/NB 差）。
   See calibration differences across models (LR good, RF/boosting/NB poor).
3. 用 **Brier 分数**量化校准。
   Quantify calibration with the Brier score.
4. 用 **Platt（sigmoid）vs 等张**校准并比较。
   Calibrate with Platt (sigmoid) vs isotonic and compare.
5. 牢记校准要用**独立校准集**（防泄漏）。
   Remember to use a held-out calibration set (no leakage).

## 目录 / TOC
1. [先建直觉：校准是什么 ⭐](#1)
2. [💳 数据 + 各模型的校准差异 ⭐](#2)
3. [Brier 分数 ⭐](#3)
4. [Platt vs 等张校准 ⭐](#4)
5. [防泄漏 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：校准是什么 ⭐ / Intuition: What Is Calibration

**校准良好**的定义：在所有模型预测"概率 ≈ 0.8"的样本里，**真的有约 80% 是正类**。换句话说，模型说的概率"言行一致"。
**Well-calibrated** means: among all samples where the model predicts "probability ≈ 0.8", **about 80% really are positive**. The stated probability matches reality.

**区分两件事**（关键）：**排序能力**（AUC 衡量，"能不能把正类排在前面"）和**校准**（"概率数值准不准"）。**两者独立**——一个模型可以 AUC 很高（排序完美）但校准很差（概率全挤在 0.4-0.6，或全推到 0/1）。
**Distinguish two things** (key): **ranking ability** (AUC, "can it order positives ahead") and **calibration** ("are the probability values accurate"). They're **independent** — a model can have high AUC (perfect ranking) yet poor calibration (probabilities all squished in 0.4-0.6, or pushed to 0/1).

**诊断工具：可靠性曲线(reliability curve)**——把预测概率分箱，x 轴是每箱的平均预测概率，y 轴是该箱的实际正类比例。**完美校准 = 贴在对角线上**。曲线在对角线**下方**说明"过度自信"（说的比实际高）。
**Diagnostic: the reliability curve** — bin predictions, x-axis = mean predicted probability per bin, y-axis = actual positive rate in that bin. **Perfect calibration = on the diagonal**. Below the diagonal = "overconfident" (claims higher than reality).


<a id="2"></a>
## 2. 数据 + 各模型的校准差异 ⭐ / Calibration Across Models

用一个合成**信用违约**数据。训几种常见模型，画它们的可靠性曲线。典型规律（面试要点）：
A synthetic **credit default** dataset. Train several common models and plot their reliability curves. Typical patterns (interview points):
- **逻辑回归**：用对数损失（=正确概率目标）训练，**天生校准好**，曲线贴对角线。
  **Logistic regression:** trained with log loss (the proper probability objective), **naturally well-calibrated**, hugging the diagonal.
- **随机森林 / boosting**：倾向把概率**推向中间或两端**，中段不准。
  **Random forest / boosting:** tend to push probabilities toward the middle or extremes, off in between.
- **朴素贝叶斯**：独立假设错 → 概率常**过度自信**（推到 0/1）。
  **Naive Bayes:** the wrong independence assumption → often **overconfident** (pushed to 0/1).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import calibration_curve
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

# 合成信用违约数据 / synthetic credit default
n = 6000
income = rng.normal(0, 1, n); debt = rng.normal(0, 1, n); age = rng.normal(0, 1, n)
logit = -1.0 + 1.5*debt - 1.0*income + 0.5*age + rng.normal(0, 0.5, n)
y = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
X = np.c_[income, debt, age]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
print(f"信用违约: {X.shape}, 违约率 {y.mean():.1%}")

models = {"逻辑回归 LogReg": LogisticRegression(max_iter=1000),
          "随机森林 RF": RandomForestClassifier(n_estimators=200, random_state=0),
          "朴素贝叶斯 NB": GaussianNB()}
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0,1],[0,1],"k--", label="完美校准 perfect")
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    # calibration_curve: 分箱后每箱(平均预测, 实际正类比例) / binned reliability points
    frac_pos, mean_pred = calibration_curve(y_te, p, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", label=name)
ax.set_xlabel("预测概率 predicted prob"); ax.set_ylabel("实际正类比例 actual rate"); ax.legend()
ax.set_title("可靠性曲线: 越贴对角线越校准好; 偏离=概率不准")
plt.tight_layout(); plt.show()
print("逻辑回归最贴对角线(对数损失训练, 天生校准); RF/NB 偏离 → 概率数值不可直接当真")


<a id="3"></a>
## 3. Brier 分数 ⭐ / The Brier Score

可靠性曲线是图，**Brier 分数**给一个数：它就是**预测概率与真实 0/1 的均方误差** $\frac1n\sum(p_i-y_i)^2$，**越低越好**。它同时奖励"排序好"和"校准好"，是衡量概率质量的标准单一指标（也可分解成校准项+区分项）。
The reliability curve is a plot; the **Brier score** gives a number: it's the **mean squared error between predicted probabilities and 0/1 labels** $\frac1n\sum(p_i-y_i)^2$, **lower is better**. It rewards both good ranking and good calibration, the standard single metric for probability quality (and decomposes into calibration + refinement terms).


In [ ]:
from sklearn.metrics import brier_score_loss, roc_auc_score

print(f"{'模型 model':<16} {'AUC(排序)':>10} {'Brier(概率质量)':>16}")
for name, m in models.items():
    p = m.predict_proba(X_te)[:, 1]
    print(f"{name:<16} {roc_auc_score(y_te, p):>10.3f} {brier_score_loss(y_te, p):>16.4f}")
print("\nAUC 衡量排序, Brier 衡量概率准不准(越低越好) — 两者独立!")
print("可能 AUC 高(排序好)但 Brier 一般(概率没校准) → 这正是需要校准的情形")


<a id="4"></a>
## 4. Platt vs 等张校准 ⭐ / Platt vs Isotonic Calibration

校准的做法：在一个**独立校准集**上，学一个"原始分数 → 校准概率"的映射。`CalibratedClassifierCV` 自动做。两种映射（面试要点）：
Calibration: on a **held-out calibration set**, learn a "raw score → calibrated probability" mapping. `CalibratedClassifierCV` does it automatically. Two mappings (interview points):
- **Platt scaling（sigmoid）**：拟合一个 sigmoid（即一个逻辑回归）。**参数少（2个）、小数据稳**，假设校准误差是 S 形。
  **Platt (sigmoid):** fit a sigmoid (a logistic regression). **Few params (2), stable on small data**, assumes an S-shaped distortion.
- **等张校准（isotonic，4.16）**：拟合一个**单调非降的分段函数**。**更灵活、能纠任意单调失真，但需更多数据**（否则过拟合校准集）。
  **Isotonic (4.16):** fit a **monotonic step function**. **More flexible, fixes any monotonic distortion, but needs more data** (else it overfits the calibration set).


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

base_rf = RandomForestClassifier(n_estimators=200, random_state=0)
# cv=5: 内部用交叉的校准折学映射(防泄漏); method 选 sigmoid 或 isotonic / calibrate via CV
cal_platt = CalibratedClassifierCV(base_rf, method="sigmoid", cv=5).fit(X_tr, y_tr)
cal_iso   = CalibratedClassifierCV(base_rf, method="isotonic", cv=5).fit(X_tr, y_tr)
raw_rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0,1],[0,1],"k--", label="完美校准 perfect")
for name, m in [("RF 未校准 raw", raw_rf), ("Platt(sigmoid)", cal_platt), ("Isotonic 等张", cal_iso)]:
    p = m.predict_proba(X_te)[:, 1]
    fp, mp = calibration_curve(y_te, p, n_bins=10, strategy="quantile")
    ax.plot(mp, fp, "o-", label=f"{name} (Brier={brier_score_loss(y_te, p):.4f})")
ax.set_xlabel("预测概率"); ax.set_ylabel("实际正类比例"); ax.legend()
ax.set_title("校准前后: Platt/等张 把 RF 概率拉回对角线 (Brier 下降)")
plt.tight_layout(); plt.show()
print("校准后曲线更贴对角线 + Brier 下降 → 概率变可信")
print("Platt: 参数少, 小数据稳; Isotonic: 更灵活但需更多数据(就是 4.16 的等张回归)")


<a id="5"></a>
## 5. 防泄漏 + 小结 ⭐ / Leakage & Summary

**关键纪律**：校准映射**必须在独立的校准集上学**，不能用训练模型的同一批数据（否则模型在那批数据上过度自信地"对"，校准会被骗）。`CalibratedClassifierCV(cv=5)` 通过交叉折自动保证这点。
**Key discipline:** the calibration mapping **must be learned on a held-out set**, not the same data the model trained on (or the model is over-confidently "right" there and fools the calibration). `CalibratedClassifierCV(cv=5)` ensures this via cross-folds.

**什么时候需要校准**：只要你**用概率值本身做决策**（按阈值定价、算期望损失 2.10、给医生一个风险数字）就需要。如果只用排序（AUC、推荐排序），则不一定需要。
**When you need it:** whenever you **use the probability value itself for decisions** (threshold-based pricing, expected loss 2.10, a risk number for a doctor). If you only use ranking (AUC, recommendation order), you may not.

```
校准良好 = 模型说"概率0.8"的样本里真有~80%是正类(言行一致)
排序(AUC) 与 校准 独立: AUC 高不代表概率准
诊断: 可靠性曲线(贴对角线=好) + Brier 分数(=概率与0/1的MSE, 越低越好)
模型校准: 逻辑回归天生好; RF/boosting/NB 常偏(NB 过度自信)
校准方法: Platt(sigmoid, 参数少小数据稳) vs 等张(灵活需更多数据, =4.16)
防泄漏: 在独立校准集上学映射(CalibratedClassifierCV(cv=5) 自动)
需要校准: 用概率值做决策时(定价/期望损失/风险数字); 只用排序时可不需
```

### 💡 面试速查 / Interview cheat-sheet
1. **校准 = 概率言行一致**(说0.8真有80%); 与排序(AUC)独立。
   Calibration = probabilities match reality; independent of ranking (AUC).
2. **逻辑回归天生校准, RF/boosting/NB 常偏**(NB 过度自信)。
   Logistic regression is naturally calibrated; RF/boosting/NB often aren't (NB overconfident).
3. **可靠性曲线(贴对角线) + Brier 分数(越低越好)** 诊断。
   Diagnose with the reliability curve (on the diagonal) + Brier score (lower better).
4. **Platt(sigmoid, 小数据稳) vs 等张(灵活, 需数据多)**。
   Platt (sigmoid, stable on small data) vs isotonic (flexible, data-hungry).
5. **校准用独立校准集**(防泄漏); 用概率做决策时才需校准。
   Calibrate on a held-out set; needed only when you use probabilities for decisions.

### 下一节 / Next
**7.10 公平性**——模型准还不够, 还要保证对不同人群(性别/种族)公平。统计平价、机会均等等公平性指标, 以及怎么缓解偏见。
**7.10 Fairness** — accuracy isn't enough; models must be fair across groups (gender/race). Fairness metrics like demographic parity and equalized odds, plus bias mitigation.
